# AI Interview Colab T4 Model Server

This notebook hosts three demo endpoints on Colab T4 GPU:

- `/stt`: Vietnamese speech-to-text with faster-whisper / PhoWhisper
- `/tts`: Vietnamese text-to-speech with VieNeu
- `/llm`: local chat model for interview turns and evaluation

Before running: `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
!nvidia-smi

## 1. Install Dependencies

This can take a few minutes. If Colab asks to restart runtime after installs, restart and run from the next cell again.

In [ ]:
!pip -q install fastapi uvicorn pyngrok python-multipart nest_asyncio requests
!pip -q install faster-whisper soundfile static-ffmpeg vieneu
!pip -q install transformers accelerate sentencepiece


## 2. Configure Models

T4 demo defaults:

- STT: `qbsmlabs/PhoWhisper-small`
- TTS: `vieneu`
- LLM: `Qwen/Qwen2.5-1.5B-Instruct` through Hugging Face Transformers

This is the light demo stack: PhoWhisper-small for Vietnamese STT, Qwen2.5 1.5B via Hugging Face for interview logic, and VieNeuTTS for Vietnamese speech output.


In [ ]:
import os

# Fast demo mode: use temporary Colab runtime storage. No Google Drive mount.
MODEL_CACHE_DIR = "/content/ai-interview-model-cache"
HF_CACHE_DIR = os.path.join(MODEL_CACHE_DIR, "huggingface")

for path in [MODEL_CACHE_DIR, HF_CACHE_DIR]:
    os.makedirs(path, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "transformers")

print("HF_HOME=", os.environ["HF_HOME"])

# Change this. Your backend must use the same token in REMOTE_MODEL_TOKEN.
AUTH_TOKEN = "demo-secret-change-me"

# STT: Vietnamese-focused small CTranslate2 model for faster-whisper.
STT_MODEL_ID = "qbsmlabs/PhoWhisper-small"
STT_COMPUTE_TYPE = "int8_float16"
STT_BEAM_SIZE = 1

# LLM: Hugging Face Transformers directly.
LLM_BACKEND = "huggingface"
HF_LLM_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MAX_INPUT_CHARS = 16000

# TTS: use GPU too. If Colab runs out of VRAM, change this back to "cpu".
TTS_DEVICE = "cuda"


## 3. Define FastAPI Model Server

In [ ]:
import os
import re
import tempfile
import threading
import time

import nest_asyncio
import requests
import torch
import uvicorn
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
from faster_whisper import WhisperModel
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer
from vieneu import Vieneu

nest_asyncio.apply()

app = FastAPI(title="Colab T4 AI Interview Model Server")

stt_model = None
tts_model = None
llm_tokenizer = None
llm_model = None


def check_auth(authorization: str | None):
    if not AUTH_TOKEN:
        return
    expected = f"Bearer {AUTH_TOKEN}"
    if authorization != expected:
        raise HTTPException(status_code=401, detail="Unauthorized")


def get_stt_model():
    global stt_model
    if stt_model is None:
        print(f"Loading STT model: {STT_MODEL_ID}")
        stt_model = WhisperModel(
            STT_MODEL_ID,
            device="cuda" if torch.cuda.is_available() else "cpu",
            compute_type=STT_COMPUTE_TYPE if torch.cuda.is_available() else "int8",
        )
    return stt_model


def get_tts_model():
    global tts_model
    if tts_model is None:
        print("Loading VieNeu TTS...")
        tts_model = Vieneu(device=TTS_DEVICE)
    return tts_model


def get_llm(model_id: str | None = None):
    global llm_tokenizer, llm_model
    model_id = model_id or HF_LLM_MODEL_ID
    if llm_model is not None and getattr(llm_model, "name_or_path", None) == model_id:
        return llm_tokenizer, llm_model

    print(f"Loading Hugging Face LLM: {model_id}")
    llm_tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if llm_tokenizer.pad_token is None:
        llm_tokenizer.pad_token = llm_tokenizer.eos_token

    llm_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )
    llm_model.eval()
    return llm_tokenizer, llm_model


def extract_json(text: str) -> str:
    text = text.strip()
    if text.startswith("{") and text.endswith("}"):
        return text
    match = re.search(r"\{.*\}", text, re.S)
    if match:
        return match.group(0)
    return text


def format_chat_prompt(tokenizer, messages: list[dict], json_mode: bool = False) -> str:
    safe_messages = messages[-12:]
    if json_mode:
        safe_messages = [
            *safe_messages,
            {"role": "user", "content": "Return ONLY valid minified JSON. Start with { and end with }. No markdown. No explanation. Escape quotes inside strings."},
        ]

    try:
        prompt = tokenizer.apply_chat_template(
            safe_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        parts = []
        for msg in safe_messages:
            role = "Assistant" if msg.get("role") == "assistant" else "User"
            if msg.get("role") == "system":
                role = "System"
            parts.append(f"{role}: {msg.get('content', '')}")
        parts.append("Assistant:")
        prompt = "\n".join(parts)

    if len(prompt) > LLM_MAX_INPUT_CHARS:
        prompt = prompt[-LLM_MAX_INPUT_CHARS:]
    return prompt


def hf_chat(messages: list[dict], temperature: float = 0.2, json_mode: bool = False, model_id: str | None = None, max_new_tokens: int = 512) -> str:
    tokenizer, model = get_llm(model_id)
    prompt = format_chat_prompt(tokenizer, messages, json_mode=json_mode)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0 and not json_mode),
            temperature=max(temperature, 0.01),
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return extract_json(text) if json_mode else text


@app.get("/health")
def health():
    return {
        "ok": True,
        "stt_model": STT_MODEL_ID,
        "stt_compute_type": STT_COMPUTE_TYPE,
        "stt_beam_size": STT_BEAM_SIZE,
        "stt_device": "cuda" if torch.cuda.is_available() else "cpu",
        "tts": "vieneu",
        "tts_device": TTS_DEVICE,
        "llm_backend": LLM_BACKEND,
        "llm_model": HF_LLM_MODEL_ID,
        "llm_device": str(next(llm_model.parameters()).device) if llm_model is not None else "not_loaded",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    }


@app.post("/stt")
async def stt(
    file: UploadFile = File(...),
    language: str = Form("vi"),
    authorization: str | None = Header(default=None),
):
    check_auth(authorization)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".webm") as tmp:
        tmp.write(await file.read())
        audio_path = tmp.name

    try:
        model = get_stt_model()
        segments, info = model.transcribe(
            audio_path,
            language=language,
            beam_size=STT_BEAM_SIZE,
            vad_filter=True,
            condition_on_previous_text=False,
        )
        text = " ".join(segment.text.strip() for segment in segments if segment.text.strip())
        return {"text": text.strip(), "language": getattr(info, "language", language)}
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        os.remove(audio_path)


class TTSRequest(BaseModel):
    text: str
    language: str = "vi"


@app.post("/tts")
async def tts(payload: TTSRequest, authorization: str | None = Header(default=None)):
    check_auth(authorization)
    model = get_tts_model()
    audio = model.infer(payload.text)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
        wav_path = tmp.name
    try:
        model.save(audio, wav_path)
        with open(wav_path, "rb") as f:
            return Response(content=f.read(), media_type="audio/wav")
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        os.remove(wav_path)


class LLMRequest(BaseModel):
    messages: list[dict]
    model: str | None = None
    json_mode: bool = False
    temperature: float = 0.2
    max_new_tokens: int = 512


@app.post("/llm")
async def llm(payload: LLMRequest, authorization: str | None = Header(default=None)):
    check_auth(authorization)
    text = await __import__("asyncio").to_thread(
        hf_chat,
        payload.messages,
        payload.temperature,
        payload.json_mode,
        payload.model,
        payload.max_new_tokens,
    )
    return {"text": text}

print("Server code loaded")


## 4. Warm Up Models

This may take several minutes and can use most of the T4 VRAM.

In [ ]:
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
get_stt_model()
get_tts_model()
get_llm()
print("Models loaded")


## 5. Expose Public URL

Optional: set your ngrok token first if you have one:

```python
ngrok.set_auth_token("ngrok_xxx")
```

In [ ]:
import subprocess, time, re, shutil

server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000),
    daemon=True,
)
server_thread.start()
time.sleep(2)

# Option A: ngrok, only if you have an authtoken.
# Put your token in Colab secrets or set it here:
# os.environ["NGROK_AUTHTOKEN"] = "ngrok_xxx"
ngrok_token = os.environ.get("NGROK_AUTHTOKEN", "").strip()

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
    public_url = ngrok.connect(8000).public_url
else:
    # Option B: Cloudflare quick tunnel, no account required.
    if not shutil.which("cloudflared"):
        !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
        !chmod +x /usr/local/bin/cloudflared

    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    public_url = None
    for _ in range(80):
        line = tunnel_proc.stdout.readline()
        print(line, end="")
        match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break
    if not public_url:
        raise RuntimeError("Could not create public tunnel")

# Keep tunnel_proc alive in the notebook global scope.
globals()["tunnel_proc"] = locals().get("tunnel_proc", None)
globals()["public_url"] = public_url

print("REMOTE_MODEL_URL=", public_url)
print("REMOTE_MODEL_TOKEN=", AUTH_TOKEN)
print("If DNS is not ready yet, wait 30-60 seconds and run the test cell again.")

## 6. Test Endpoints

Copy the printed `REMOTE_MODEL_URL` and `REMOTE_MODEL_TOKEN` into your backend `.env`:

```env
REMOTE_MODEL_URL=https://xxxx.ngrok-free.app
REMOTE_MODEL_TOKEN=demo-secret-change-me

STT_PROVIDER=remote
TTS_PROVIDER=remote
LLM_PROVIDER=remote
EVALUATOR_PROVIDER=remote
REMOTE_LLM_MODEL=
CV_LLM_PROVIDER=remote
PHOWHISPER_ENABLED=false
```

Then restart your local backend.

In [ ]:
import requests, time

headers = {"Authorization": f"Bearer {AUTH_TOKEN}"}

print("Local health:")
print(requests.get("http://127.0.0.1:8000/health", headers=headers, timeout=10).json())

print("Public tunnel health:")
last_error = None
for attempt in range(1, 13):
    try:
        res = requests.get(f"{public_url}/health", headers=headers, timeout=15)
        print(res.json())
        break
    except Exception as exc:
        last_error = exc
        print(f"Attempt {attempt}/12 failed: {exc}")
        time.sleep(5)
else:
    raise last_error

payload = {
    "messages": [
        {"role": "system", "content": "You are Alex, a Vietnamese technical interviewer. Answer concisely in Vietnamese."},
        {"role": "user", "content": "Chào bạn, hãy hỏi tôi một câu phỏng vấn Python."},
    ],
    "temperature": 0.2,
    "max_new_tokens": 160,
}
print(requests.post(f"{public_url}/llm", json=payload, headers=headers, timeout=180).json())

# Keep REMOTE_LLM_MODEL blank so this endpoint uses the Hugging Face Qwen2.5 default.
